# Discrete kinematics: technical pilot

30 selected participants, 15 healthy and 15 injured. This notebook communicates checked source values and technical quality only. No clinical group-effect estimates or diagnostic models are fit. Source scalars are preserved, including negative angles. Event detection and valid cycle counts remain unverified.

Run `scripts/run_pilot.py` before using this notebook; raw files are processed sequentially by reusable src functions.

In [1]:
from pathlib import Path
import sys, json
import pandas as pd
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.biomechanics.feature_registry import build_registry
registry, placeholders = build_registry(ROOT)
features = pd.read_csv(ROOT / 'data/processed/pilot_features_long.csv', dtype={'participant_id': 'string'})
qc = pd.read_csv(ROOT / 'reports/tables/pilot_qc_trials.csv', dtype={'participant_id': 'string'})
summary = json.loads((ROOT / 'reports/tables/pilot_summary.json').read_text())

In [2]:
{key: summary[key] for key in ['participants', 'features_per_side', 'finite_source_values', 'provisionally_available_values', 'structural_pass_trials', 'independently_validated_events_trials']}

{'participants': 30, 'features_per_side': 13, 'finite_source_values': 780, 'provisionally_available_values': 780, 'structural_pass_trials': 30, 'independently_validated_events_trials': 0}

## Verified definitions and units

Stance-normalized peaks are defined by the original source code. Knee flexion uses the first 70 of 101 stance samples; hip adduction uses the first 60. STRIDE_RATE is strides/min, not Hz or steps/min. The listed code definitions do not independently validate the historical processing of every JSON.

In [3]:
pd.DataFrame(registry)[['feature', 'unit', 'phase', 'definition', 'source_line_start']]

,feature,unit,phase,definition,source_line_start
0,STRIDE_RATE,strides/min,successive ipsilateral touchdowns,Median of 60*hz/diff(TD); not steps/min and no...,776
1,STRIDE_LENGTH,m,successive ipsilateral touchdowns,Median of source heel-derived speed * diff(TD)/hz,782
2,SWING_TIME,s,toe-off to next ipsilateral touchdown,Median swing-frame interval / hz,788
3,STANCE_TIME,s,touchdown to toe-off,Median stance-frame interval / hz,802
4,ANKLE_DF_PEAK_ANGLE,deg,whole normalized stance,Negative median of per-stance minimum axis-3 a...,848
5,ANKLE_EVE_PEAK_ANGLE,deg,whole normalized stance,Left median minimum axis 1; right negative med...,869
6,ANKLE_ROT_PEAK_ANGLE,deg,whole normalized stance,Preserve exact source left/right rotation conv...,970
7,KNEE_FLEX_PEAK_ANGLE,deg,first 70 of 101 normalized stance samples,Negative median of per-stance axis-3 maximum i...,998
8,KNEE_ADD_PEAK_ANGLE,deg,whole normalized stance,Left median maximum axis 1; right negative med...,1019
9,KNEE_ROT_PEAK_ANGLE,deg,whole normalized stance,Left median maximum axis 2; right negative med...,1063


In [4]:
features[['participant_id', 'side', 'feature', 'value', 'unit', 'value_qc', 'analysis_value', 'events_qc']].head(15)

,participant_id,side,feature,value,unit,value_qc,analysis_value,events_qc
0,101111,left,STRIDE_RATE,80.536913,strides/min,source_scalar_checked,80.536913,unverified
1,101111,left,STRIDE_LENGTH,2.141929,m,source_scalar_checked,2.141929,unverified
2,101111,left,SWING_TIME,0.462500,s,source_scalar_checked,0.462500,unverified
3,101111,left,STANCE_TIME,0.280000,s,source_scalar_checked,0.280000,unverified
4,101111,left,ANKLE_DF_PEAK_ANGLE,20.749535,deg,source_scalar_checked,20.749535,unverified
5,101111,left,ANKLE_EVE_PEAK_ANGLE,-5.211766,deg,source_scalar_checked,-5.211766,unverified
6,101111,left,ANKLE_ROT_PEAK_ANGLE,3.036545,deg,source_scalar_checked,3.036545,unverified
7,101111,left,KNEE_FLEX_PEAK_ANGLE,-40.192099,deg,source_scalar_checked,-40.192099,unverified
8,101111,left,KNEE_ADD_PEAK_ANGLE,-2.544707,deg,source_scalar_checked,-2.544707,unverified
9,101111,left,KNEE_ROT_PEAK_ANGLE,-18.384889,deg,source_scalar_checked,-18.384889,unverified


## Technical quality and coverage

Each participant contributes two side-specific slots per feature. Slots are not independent people. Source fields that are unassigned placeholders are excluded; exactly-zero selected angles remain preserved and require review.

In [5]:
features.groupby(['feature', 'unit']).agg(expected_slots=('side', 'size'), finite_source_values=('value', 'count'), available_slots=('provisionally_available', 'sum'))

,,expected_slots,finite_source_values,available_slots
feature,unit,,,
ANKLE_DF_PEAK_ANGLE,deg,60,60,60
ANKLE_EVE_PEAK_ANGLE,deg,60,60,60
ANKLE_ROT_PEAK_ANGLE,deg,60,60,60
HIP_ADD_PEAK_ANGLE,deg,60,60,60
HIP_EXT_PEAK_ANGLE,deg,60,60,60
HIP_ROT_PEAK_ANGLE,deg,60,60,60
KNEE_ADD_PEAK_ANGLE,deg,60,60,60
KNEE_FLEX_PEAK_ANGLE,deg,60,60,60
KNEE_ROT_PEAK_ANGLE,deg,60,60,60


In [6]:
qc[['participant_id', 'qc_status', 'sampling_hz', 'time_span_s', 'provisionally_available_slots', 'events_qc']]

,participant_id,qc_status,sampling_hz,time_span_s,provisionally_available_slots,events_qc
0,101111,structural_pass,200,24.995,26,not_available_without_official_reprocessing
1,100288,structural_pass,200,19.145,26,not_available_without_official_reprocessing
2,200244,structural_pass,200,24.995,26,not_available_without_official_reprocessing
3,101067,structural_pass,200,24.995,26,not_available_without_official_reprocessing
4,100279,structural_pass,200,12.455,26,not_available_without_official_reprocessing
5,101577,structural_pass,200,24.995,26,not_available_without_official_reprocessing
6,101302,structural_pass,200,59.995,26,not_available_without_official_reprocessing
7,102009,structural_pass,200,59.995,26,not_available_without_official_reprocessing
8,100246,structural_pass,200,19.995,26,not_available_without_official_reprocessing
9,100504,structural_pass,200,59.995,26,not_available_without_official_reprocessing


![Technical pilot quality](../reports/figures/pilot_quality_overview.png)

See `reports/milestone_02_pilot_summary.md` and `docs/official_processing_review.md` for traceability, engineering review triggers and remaining event/reconstruction validation. Later waveform inference, phenotyping and classification remain separate stages.